# Дообучение YOLO на SportsMOT

In [11]:
!pip -q install -U ultralytics pyyaml tqdm

In [ ]:
import os
from pathlib import Path
import shutil
import configparser
from collections import defaultdict
import yaml
from tqdm.auto import tqdm

#kaggle
SPORTSMOT_ROOT = Path("/kaggle/input/datasets/username/sportsmot")
WORKDIR = Path("/kaggle/working")
YOLO_ROOT = Path("/kaggle/working/sportsmot_yolo")

MODEL_NAME = "yolo11s.pt"
EPOCHS = 20
IMGSZ = 960
BATCH = 16
DEVICE = 0              
WORKERS = 4

CLASS_ID = 0
CLASS_NAME = "player"

IMAGE_MODE = "symlink" 

assert SPORTSMOT_ROOT.exists(), f"Не найден SPORTSMOT_ROOT: {SPORTSMOT_ROOT}"
YOLO_ROOT.mkdir(parents=True, exist_ok=True)
print("SPORTSMOT_ROOT =", SPORTSMOT_ROOT)
print("YOLO_ROOT      =", YOLO_ROOT)


Для обучения детектора мы используем только `train` и `val`.  
Тестовые данные не трогаем.


In [13]:
def read_split_file(path: Path):
    lines = [x.strip() for x in path.read_text(encoding="utf-8").splitlines()]
    return [x for x in lines if x]

def parse_seqinfo(seqinfo_path: Path):
    cfg = configparser.ConfigParser()
    cfg.read(seqinfo_path)
    seq = cfg["Sequence"]
    return {
        "name": seq.get("name"),
        "imDir": seq.get("imDir", "img1"),
        "frameRate": seq.getint("frameRate", fallback=25),
        "seqLength": seq.getint("seqLength"),
        "imWidth": seq.getint("imWidth"),
        "imHeight": seq.getint("imHeight"),
        "imExt": seq.get("imExt", ".jpg"),
    }

def mot_line_to_yolo(parts, img_w, img_h):
    # frame, track_id, bb_left, bb_top, bb_width, bb_height, conf, class, visibility
    frame_id = int(float(parts[0]))
    track_id = int(float(parts[1]))
    x = float(parts[2])
    y = float(parts[3])
    w = float(parts[4])
    h = float(parts[5])

    # 7-й  = mark/conf, 8-й = class, 9-й = visibility
    mark = int(float(parts[6])) if len(parts) > 6 else 1
    obj_class = int(float(parts[7])) if len(parts) > 7 else 1
    visibility = float(parts[8]) if len(parts) > 8 else 1.0

    # - только размеченные объекты
    # - bbox > 0
    # - только class==1 (human/player)
    if mark <= 0:
        return None
    if w <= 1 or h <= 1:
        return None
    if obj_class != 1:
        return None

    # clip к размеру изображения
    x1 = max(0.0, min(x, img_w - 1))
    y1 = max(0.0, min(y, img_h - 1))
    x2 = max(0.0, min(x + w, img_w))
    y2 = max(0.0, min(y + h, img_h))

    w = x2 - x1
    h = y2 - y1
    if w <= 1 or h <= 1:
        return None

    xc = x1 + w / 2.0
    yc = y1 + h / 2.0

    # YOLO format: class xc yc w h (all normalized)
    return frame_id, f"{CLASS_ID} {xc / img_w:.6f} {yc / img_h:.6f} {w / img_w:.6f} {h / img_h:.6f}"

def ensure_parent(path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)

def link_or_copy(src: Path, dst: Path, mode: str = "copy"):
    ensure_parent(dst)
    if dst.exists():
        return
    if mode == "symlink":
        try:
            os.symlink(src, dst)
        except FileExistsError:
            pass
        except OSError:
            shutil.copy2(src, dst)
    else:
        shutil.copy2(src, dst)

In [14]:
def convert_split(mot_split_dir: Path, out_root: Path, split_name: str, image_mode: str = "copy"):
    out_images = out_root / "images" / split_name
    out_labels = out_root / "labels" / split_name
    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)

    seq_dirs = sorted([p for p in mot_split_dir.iterdir() if p.is_dir()])
    print(f"Converting split={split_name!r}, sequences={len(seq_dirs)}")

    n_images = 0
    n_labels = 0
    n_boxes = 0

    for seq_dir in tqdm(seq_dirs):
        seqinfo = parse_seqinfo(seq_dir / "seqinfo.ini")
        img_w, img_h = seqinfo["imWidth"], seqinfo["imHeight"]
        img_dir = seq_dir / seqinfo["imDir"]
        gt_path = seq_dir / "gt" / "gt.txt"

        
        labels_by_frame = defaultdict(list)
        if gt_path.exists():
            for raw in gt_path.read_text(encoding="utf-8").splitlines():
                raw = raw.strip()
                if not raw:
                    continue
                parts = [x.strip() for x in raw.split(",")]
                yolo_obj = mot_line_to_yolo(parts, img_w, img_h)
                if yolo_obj is None:
                    continue
                frame_id, yolo_line = yolo_obj
                labels_by_frame[frame_id].append(yolo_line)

    
        image_files = sorted(img_dir.glob("*"))
        for img_path in image_files:
            if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp"}:
                continue

            frame_id = int(img_path.stem)
            
            new_stem = f"{seq_dir.name}_{img_path.stem}"
            out_img_path = out_images / f"{new_stem}{img_path.suffix.lower()}"
            out_lbl_path = out_labels / f"{new_stem}.txt"

            link_or_copy(img_path, out_img_path, mode=image_mode)
            ensure_parent(out_lbl_path)
            out_lbl_path.write_text("\n".join(labels_by_frame.get(frame_id, [])), encoding="utf-8")

            n_images += 1
            n_labels += 1
            n_boxes += len(labels_by_frame.get(frame_id, []))

    print(f"[{split_name}] images={n_images}, label_files={n_labels}, boxes={n_boxes}")

In [ ]:
train_dir = SPORTSMOT_ROOT / "dataset" / "train"
val_dir = SPORTSMOT_ROOT / "dataset" / "val"

assert train_dir.exists(), f"Не найден каталог: {train_dir}"
assert val_dir.exists(), f"Не найден каталог: {val_dir}"

convert_split(train_dir, YOLO_ROOT, "train", image_mode=IMAGE_MODE)
convert_split(val_dir, YOLO_ROOT, "val", image_mode=IMAGE_MODE)

In [16]:
data_yaml = {
    "path": str(YOLO_ROOT),
    "train": "images/train",
    "val": "images/val",
    "names": {
        0: CLASS_NAME
    }
}

yaml_path = YOLO_ROOT / "sportsmot.yaml"
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml, f, allow_unicode=True, sort_keys=False)

print(yaml_path.read_text(encoding="utf-8"))

path: /kaggle/working/sportsmot_yolo
train: images/train
val: images/val
names:
  0: player



In [ ]:
for p in [
    YOLO_ROOT / "images" / "train",
    YOLO_ROOT / "labels" / "train",
    YOLO_ROOT / "images" / "val",
    YOLO_ROOT / "labels" / "val",
    YOLO_ROOT / "sportsmot.yaml",
]:
    print(p, "->", p.exists())

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_NAME)

results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    project=str(YOLO_ROOT / "runs"),
    name="yolo_sportsmot_finetune",
    pretrained=True,
    exist_ok=True,
    cache=False,
    close_mosaic=10,
)

In [ ]:

# best_model = YOLO(str(YOLO_ROOT / "runs" / "yolo_sportsmot_finetune" / "weights" / "best.pt"))
# metrics = best_model.val(data=str(yaml_path), split="val")
# metrics

In [ ]:

# sample_images = sorted((YOLO_ROOT / "images" / "val").glob("*.jpg"))[:8]
# sample_images = [str(p) for p in sample_images]

# pred_model = YOLO(str(YOLO_ROOT / "runs" / "yolo_sportsmot_finetune" / "weights" / "best.pt"))
# pred_model.predict(
#     source=sample_images,
#     imgsz=IMGSZ,
#     conf=0.25,
#     save=True,
#     project=str(YOLO_ROOT / "runs"),
#     name="pred_examples",
#     exist_ok=True,
# )